# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Which of the three lanes did you choose (or freestyle)? Why is it worth solving? What is the specific output of your model?*

**Lane 2 — Refresh / Content Opportunity Scoring**

I chose this lane because it connects the available page-level search and content signals to a practical prioritization decision. The aim is not simply to predict whether a page is declining. Instead, the project will help an SEO or content reviewer decide which pseudonymized content pages may deserve attention first when review time is limited. The proposed output is a ranked review queue containing a score, reason codes, a confidence level, and a possible action such as refresh, expand, consolidate, prune, or monitor.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Research Question
Which content pages should an SEO specialist or content editor prioritize for manual review based on their observed search performance, content age, and engagement signals?

### Framework for Decision Support
*   **Decision:** Which content pages should be reviewed first when editorial resources are limited?
*   **Stakeholder:** An SEO specialist, content strategist, or content editor responsible for website maintenance.
*   **Action:** Review the recommended pages and determine whether they should be refreshed, expanded, consolidated, pruned, protected, or monitored.
*   **Cost of a False Positive:** Editorial time may be spent reviewing or modifying a page that did not require intervention. Unnecessary changes could also affect a page that was performing acceptably.
*   **Cost of a False Negative:** A potentially important declining page may not be reviewed, allowing it to continue losing search visibility or traffic.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks*

The code below loads the starter CSV and calculates three simple opportunity groups using transparent definitions from the starter workflow. These are exploratory counts, not labels proving that an intervention will succeed.

In [9]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

if "google.colab" in sys.modules:
    repo_url = "https://github.com/pretom26/ml_internship_flyrankAI.git"
    repo_dir = Path("/content/ml_internship_flyrankAI")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)
    os.chdir(repo_dir)
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Dataset not found.")

data_path = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

declining_with_demand = (df["trend_direction"].eq("down") & df["impressions_90d"].ge(100))
low_ctr_visible = (df["impressions_90d"].ge(500) & df["avg_position"].gt(0) & df["avg_position"].le(20) & df["ctr"].lt(0.5))
page_one_decay_risk = (df["avg_position"].gt(0) & df["avg_position"].le(10) & df["content_age_days"].ge(180))

evidence = pd.DataFrame({
    "signal": [
        "Declining with demand",
        "Visible but CTR below 0.5%",
        "Page-one and at least 180 days old",
    ],
    "definition": [
        "Trend down & impressions >= 100",
        "Top 20 pos, 500+ impressions, CTR < 0.5%",
        "Top 10 pos & age >= 180 days",
    ],
    "page_count": [
        int(declining_with_demand.sum()),
        int(low_ctr_visible.sum()),
        int(page_one_decay_risk.sum()),
    ],
})

evidence["share_of_30k_pages"] = (evidence["page_count"] / len(df) * 100).round(1).astype(str) + "%"

print(f"Dataset scope: {len(df):,} pages across {df['client_id'].nunique()} clients.")
display(evidence)

Dataset scope: 30,000 pages across 32 clients.


,signal,definition,page_count,share_of_30k_pages
0,Declining with demand,Trend down & impressions >= 100,13152,43.8%
1,Visible but CTR below 0.5%,"Top 20 pos, 500+ impressions, CTR < 0.5%",9759,32.5%
2,Page-one and at least 180 days old,Top 10 pos & age >= 180 days,7076,23.6%


### Why these numbers support the lane

The dataset covers 30,000 pages across 32 pseudonymized clients. The initial screening identified several large groups of possible review candidates:

*   **43.8% or 13,152 pages** show a downward trend while still receiving at least 100 impressions.
*   **32.5% or 9,759 pages** have at least 500 impressions, rank between positions 1 and 20, and have a CTR below 0.5%.
*   **23.6% or 7,076 pages** are at least 180 days old while still appearing within the top 10 positions.

These groups overlap and should not be added together. However, their size indicates that the number of possible review candidates is much greater than a content team could reasonably inspect manually. This supports exploring a ranked review queue that prioritizes pages using several observable signals while also providing understandable reasons for each recommendation.

## 4. Careful words: what I can and cannot claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What this project may provide
This project aims to create a directional decision-support system based on observed patterns in the starter data. It may help an SEO specialist or content editor identify pages that warrant earlier manual review. The resulting score or ranking will indicate review priority rather than prove that a particular page requires an update.

The project will compare the proposed ranking approach with a transparent rule-based baseline. It will only claim improvement if the evaluation results show that the proposed method produces a more useful review queue under the selected evaluation criteria.

### What this project cannot claim
*   **No causal claim:** The analysis cannot prove that refreshing a recommended page will cause its traffic or ranking to improve.
*   **No guarantee of recovery:** A highly ranked review candidate may not benefit from an update.
*   **No prediction of Google’s algorithm:** The project will not attempt to reverse-engineer search algorithms or guarantee future rankings.
*   **No automatic editorial action:** The output will support human review. Final decisions must consider context such as seasonality, search intent, brand requirements, and business priorities.

In summary, the proposed system is intended to prioritize human review using observed data, not to make guaranteed or autonomous content decisions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.